In [ ]:
!python -m pip install pyyaml==5.1
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities (e.g. compiled operators).
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

# Properly install detectron2. (Please do not install twice in both ways)
# !python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

In [ ]:
print(torch.cuda.is_available())

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()
import warnings
warnings.filterwarnings(action='ignore')

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.structures import BoxMode
import random
import os
import json
import numpy as np

from detectron2.data import MetadataCatalog
import torch
from torchvision.transforms.functional import resize

In [ ]:
# via_region_data.json → Detectron2 input 포맷으로 변환 함수 (템플릿 그대로)
def get_balloon_dicts(img_dir):
    json_file = os.path.join(img_dir, "via_region_data.json")
    with open(json_file) as f:
        imgs_anns = json.load(f)
    dataset_dicts = []
    for idx, v in enumerate(imgs_anns.values()):
        record = {}
        filename = os.path.join(img_dir, v["filename"])
        height, width = cv2.imread(filename).shape[:2]
        record["file_name"] = filename
        record["image_id"]  = idx
        record["height"]    = height
        record["width"]     = width
        objs = []
        for _, anno in v["regions"].items():
            anno = anno["shape_attributes"]
            px, py = anno["all_points_x"], anno["all_points_y"]
            poly = [(x+0.5, y+0.5) for x,y in zip(px,py)]
            poly = [p for xy in poly for p in xy]
            objs.append({
                "bbox": [np.min(px), np.min(py), np.max(px), np.max(py)],
                "bbox_mode": BoxMode.XYXY_ABS,
                "segmentation": [poly],
                "category_id": 0,
            })
        record["annotations"] = objs
        dataset_dicts.append(record)
    return dataset_dicts

# DatasetCatalog 등록
for d in ["train","val"]:
    name = f"balloon_{d}"
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
    DatasetCatalog.register(name, lambda d=d: get_balloon_dicts(f"/content/balloon/{d}"))
    MetadataCatalog.get(name).set(thing_classes=["balloon"])
balloon_metadata = MetadataCatalog.get("balloon_train")

In [ ]:
cfg = get_cfg()
model_name = "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
cfg.merge_from_file(model_zoo.get_config_file(model_name))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(model_name)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

In [ ]:
# 원본 해상도 유지 (AssertionError 방지)
d0 = random.choice(get_balloon_dicts("/content/balloon/train"))
im0 = cv2.imread(d0["file_name"])
h0, w0 = im0.shape[:2]
cfg.INPUT.MIN_SIZE_TEST = h0
cfg.INPUT.MAX_SIZE_TEST = max(h0,w0)

In [ ]:
# Predictor 생성 및 inference
predictor_base = DefaultPredictor(cfg)
outputs0 = predictor_base(im0)

In [ ]:
# COCO 클래스(80개)용 메타데이터로 시각화
coco_meta = MetadataCatalog.get("coco_2017_val")
v0 = Visualizer(im0[:, :, ::-1], metadata=coco_meta, scale=1.0)
out0 = v0.draw_instance_predictions(outputs0["instances"].to("cpu"))
cv2_imshow(out0.get_image()[:, :, ::-1])

In [ ]:
'''Don't modify '''
# 풍선 데이터셋의 형식을 파인튜닝할 수 있게 만들어주는 함수입니다.
def get_balloon_dicts(img_dir):
    json_file = os.path.join(img_dir, "via_region_data.json")
    with open(json_file) as f:
        imgs_anns = json.load(f)

    dataset_dicts = []
    for idx, v in enumerate(imgs_anns.values()):
        record = {}
        filename = os.path.join(img_dir, v["filename"])
        height, width = cv2.imread(filename).shape[:2]
        record["file_name"] = filename
        record["image_id"] = idx
        record["height"] = height
        record["width"] = width
        annos = v["regions"]
        objs = []
        for _, anno in annos.items():
            assert not anno["region_attributes"]
            anno = anno["shape_attributes"]
            px = anno["all_points_x"]
            py = anno["all_points_y"]
            poly = [(x + 0.5, y + 0.5) for x, y in zip(px, py)]
            poly = [p for x in poly for p in x]
            obj = {
                "bbox": [np.min(px), np.min(py), np.max(px), np.max(py)],
                "bbox_mode": BoxMode.XYXY_ABS,
                "segmentation": [poly],
                "category_id": 0,
            }
            objs.append(obj)
        record["annotations"] = objs
        dataset_dicts.append(record)
    return dataset_dicts

'''Don't modify '''
# 풍선 데이터셋을 DatasetCatalog에 등록하는 과정입니다.
for d in ["train", "val"]:
    if "balloon_" + d in DatasetCatalog.list():
        DatasetCatalog.remove("balloon_" + d)
    DatasetCatalog.register("balloon_" + d, lambda d=d: get_balloon_dicts("balloon/" + d))
    MetadataCatalog.get("balloon_" + d).set(thing_classes=["balloon"])
balloon_metadata = MetadataCatalog.get("balloon_train")
dataset_dicts = get_balloon_dicts("balloon/train")

# 풍선 데이터셋을 pre-trained 모델에 넣어 확인해보는 과정입니다.
'''
d["file_name"]으로부터 파일명만 추출하여 저장하고 싶을 경우
import os
filename = os.path.basename(d["file_name"])
cv2.imwrite(f'./yourroot/{filename}',out.get_image()[:, :, ::-1])
'''

random.seed(1234)
for d in random.sample(dataset_dicts, 15):
    im = cv2.imread(d["file_name"])
    outputs = predictor(im)  # format is documented at https://detectron2.readthedocs.io/tutorials/models.html#model-output-format
    v = Visualizer(im[:, :, ::-1],
                   MetadataCatalog.get(cfg.DATASETS.TRAIN[0]),
                   scale=0.5
    )
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

In [ ]:
# 3-2
###Finetuning #####

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(model_name))
cfg.DATASETS.TRAIN = ("balloon_train",)
cfg.DATASETS.TEST  = ()            # 평가 파트는 별도에서 실행할 예정이므로 빈 튜플
cfg.DATALOADER.NUM_WORKERS = 2

# COCO 사전학습 가중치로 초기화
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(model_name)

#
cfg.SOLVER.IMS_PER_BATCH = 2       # 배치 사이즈
cfg.SOLVER.BASE_LR       = 0.00025 # 학습률
cfg.SOLVER.MAX_ITER      = 300     # 총 반복 횟수

cfg.SOLVER.STEPS = []               # lr 스케줄링 포인트
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NUM_CLASSES        = 1

# 출력 디렉토리 설정
cfg.OUTPUT_DIR = "/content/balloon/output"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Trainer 정의 및 학습 시작
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:
# === 3-2.5 Fine-tuned 모델 예시 시각화 ===
# 최종 가중치 불러와 predictor 재생성
'''Don't modify '''
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
predictor_ft = DefaultPredictor(cfg)

# val 에서 랜덤 3장 뽑아 시각화
for d in random.sample(get_balloon_dicts("/content/balloon/val"), 3):
    im = cv2.imread(d["file_name"])
    out = predictor_ft(im)
    v = Visualizer(im[:, :, ::-1], metadata=balloon_metadata, scale=1.0)
    vis = v.draw_instance_predictions(out["instances"].to("cpu"))
    cv2_imshow(vis.get_image()[:, :, ::-1])

In [ ]:
# 3-3
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

evaluator  = COCOEvaluator("balloon_val", cfg, False, output_dir=cfg.OUTPUT_DIR)
val_loader = build_detection_test_loader(cfg, "balloon_val")
metrics    = inference_on_dataset(trainer.model, val_loader, evaluator)
print(metrics)

In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

evaluator  = COCOEvaluator("balloon_val", cfg, False, output_dir=cfg.OUTPUT_DIR)
val_loader = build_detection_test_loader(cfg, "balloon_val")
metrics    = inference_on_dataset(trainer.model, val_loader, evaluator)
print(metrics)

In [ ]:
# 3-4
# 1) Fine-tuned 모델 가중치 로드
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # 필요 시 threshold 조정

# 2) Fine-tuned predictor 재생성
predictor_ft = DefaultPredictor(cfg)

# 3) 3-1에서 사용한 im0 이미지로 inference
outputs_ft0 = predictor_ft(im0)

# 4) balloon_metadata를 사용해 시각화
v_ft0 = Visualizer(im0[:, :, ::-1], metadata=balloon_metadata, scale=1.0)
out_ft0 = v_ft0.draw_instance_predictions(outputs_ft0["instances"].to("cpu"))

# 5) 결과 출력
cv2_imshow(out_ft0.get_image()[:, :, ::-1])